In [1]:
import pandas as pd
import numpy as np
import os

from catboost import CatBoostRegressor

BASE_DIR = r"C:\Users\vikra\Desktop\Supply Chain Algo"

DATA_DIR = os.path.join(BASE_DIR, "Data")
RAW_DIR = os.path.join(DATA_DIR, "Raw")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")

MODEL_DIR = os.path.join(BASE_DIR, "models")
REPORT_DIR = os.path.join(BASE_DIR, "reports")

os.makedirs(REPORT_DIR, exist_ok=True)

# Load optimized CatBoost model
model_path = os.path.join(
    MODEL_DIR,
    "optimized_catboost_model.cbm"
)

model = CatBoostRegressor()
model.load_model(model_path)

print("Optimized CatBoost model loaded!")

Optimized CatBoost model loaded!


In [2]:
# Load datasets
history = pd.read_csv(
    os.path.join(PROCESSED_DIR, "sales_clean.csv")
)

future = pd.read_csv(
    os.path.join(RAW_DIR, "test.csv")
)

stores = pd.read_csv(
    os.path.join(PROCESSED_DIR, "stores_clean.csv")
)

holidays = pd.read_csv(
    os.path.join(RAW_DIR, "holidays_events.csv")
)

# Normalize column names
for data in [history, future, stores, holidays]:
    data.columns = data.columns.str.strip().str.lower()

# Convert date columns
history["date"] = pd.to_datetime(history["date"])
future["date"] = pd.to_datetime(future["date"])
holidays["date"] = pd.to_datetime(holidays["date"])

print("Historical data:", history.shape)
print("Future data:", future.shape)
print("Stores:", stores.shape)
print("Holidays:", holidays.shape)

print("\nFuture date range:")
print(future["date"].min(), "to", future["date"].max())

display(future.head())

Historical data: (3000888, 21)
Future data: (28512, 5)
Stores: (54, 5)
Holidays: (350, 6)

Future date range:
2017-08-16 00:00:00 to 2017-08-31 00:00:00


,id,date,store_nbr,family,onpromotion
0,3000888,2017-08-16,1,AUTOMOTIVE,0
1,3000889,2017-08-16,1,BABY CARE,0
2,3000890,2017-08-16,1,BEAUTY,2
3,3000891,2017-08-16,1,BEVERAGES,20
4,3000892,2017-08-16,1,BOOKS,0


In [3]:
required_columns = [
    "id",
    "date",
    "store_nbr",
    "family",
    "onpromotion"
]

missing = [
    col for col in required_columns
    if col not in future.columns
]

assert not missing, f"Missing test columns: {missing}"

print("Future dataset validation successful!")

Future dataset validation successful!


In [4]:
history = history[
    ["date", "store_nbr", "family", "sales"]
].copy()

history["sales"] = pd.to_numeric(
    history["sales"],
    errors="coerce"
)

history = history.dropna(
    subset=["date", "store_nbr", "family", "sales"]
)

history["store_nbr"] = history["store_nbr"].astype(int)
history["family"] = history["family"].astype(str)

history = history.sort_values(
    ["store_nbr", "family", "date"]
).reset_index(drop=True)

# Validate unique records
duplicates = history.duplicated(
    ["date", "store_nbr", "family"]
).sum()

print("Historical duplicates:", duplicates)

assert duplicates == 0, "Resolve duplicate records before forecasting."

Historical duplicates: 0


In [5]:
history_sales = {}

for (store, family), group in history.groupby(
    ["store_nbr", "family"]
):
    history_sales[(int(store), str(family))] = (
        group.sort_values("date")["sales"].tolist()
    )

print("Historical store-category combinations:",
      len(history_sales))

Historical store-category combinations: 1782


In [6]:
# Clean store information
store_lookup = stores[
    ["store_nbr", "city", "state"]
].drop_duplicates().copy()

store_lookup["store_nbr"] = (
    store_lookup["store_nbr"].astype(int)
)

store_lookup["city_key"] = (
    store_lookup["city"].astype(str).str.strip().str.casefold()
)

store_lookup["state_key"] = (
    store_lookup["state"].astype(str).str.strip().str.casefold()
)

# Clean holiday information
holiday_data = holidays.copy()

holiday_data["locale"] = (
    holiday_data["locale"]
    .astype(str)
    .str.strip()
    .str.casefold()
)

holiday_data["locale_name"] = (
    holiday_data["locale_name"]
    .astype(str)
    .str.strip()
    .str.casefold()
)

# Exclude events explicitly marked as transferred
if "transferred" in holiday_data.columns:
    transferred = (
        holiday_data["transferred"]
        .astype(str)
        .str.strip()
        .str.casefold()
        .isin(["true", "1", "yes"])
    )

    holiday_data = holiday_data[~transferred].copy()

In [7]:
holiday_records = []

for _, event in holiday_data.iterrows():

    event_date = event["date"]
    locale = event["locale"]
    locale_name = event["locale_name"]

    if locale == "national":

        matched_stores = store_lookup

    elif locale == "regional":

        matched_stores = store_lookup[
            store_lookup["state_key"] == locale_name
        ]

    elif locale == "local":

        matched_stores = store_lookup[
            store_lookup["city_key"] == locale_name
        ]

    else:
        continue

    for store_number in matched_stores["store_nbr"]:
        holiday_records.append({
            "date": event_date,
            "store_nbr": store_number
        })

holiday_store = pd.DataFrame(holiday_records)

if not holiday_store.empty:

    holiday_store = (
        holiday_store
        .groupby(["date", "store_nbr"])
        .size()
        .reset_index(name="holiday_event_count")
    )

    holiday_store["has_holiday_event"] = 1

else:

    holiday_store = pd.DataFrame(
        columns=[
            "date",
            "store_nbr",
            "holiday_event_count",
            "has_holiday_event"
        ]
    )

print("Holiday mapping created!")
display(holiday_store.head())

Holiday mapping created!


,date,store_nbr,holiday_event_count,has_holiday_event
0,2012-03-02,52,1,1
1,2012-03-02,53,1,1
2,2012-04-01,12,1,1
3,2012-04-01,13,1,1
4,2012-04-12,37,1,1


In [8]:
future = future.merge(
    holiday_store,
    on=["date", "store_nbr"],
    how="left",
    validate="many_to_one"
)

future["holiday_event_count"] = (
    future["holiday_event_count"].fillna(0)
)

future["has_holiday_event"] = (
    future["has_holiday_event"].fillna(0)
)

print("Future holiday features added!")

Future holiday features added!


In [9]:
future["year"] = future["date"].dt.year
future["month"] = future["date"].dt.month
future["day"] = future["date"].dt.day

future["day_of_week"] = (
    future["date"].dt.dayofweek
)

future["day_of_year"] = (
    future["date"].dt.dayofyear
)

future["week_of_year"] = (
    future["date"].dt.isocalendar().week.astype(int)
)

future["quarter"] = future["date"].dt.quarter

future["is_weekend"] = (
    future["day_of_week"] >= 5
).astype(int)

future["month_sin"] = np.sin(
    2 * np.pi * future["month"] / 12
)

future["month_cos"] = np.cos(
    2 * np.pi * future["month"] / 12
)

future["day_sin"] = np.sin(
    2 * np.pi * future["day_of_week"] / 7
)

future["day_cos"] = np.cos(
    2 * np.pi * future["day_of_week"] / 7
)

future["onpromotion"] = pd.to_numeric(
    future["onpromotion"],
    errors="coerce"
).fillna(0)

print("Future calendar features created!")

Future calendar features created!


In [10]:
categorical_features = [
    "store_nbr",
    "family"
]

numeric_features = [
    "onpromotion",
    "has_holiday_event",
    "holiday_event_count",
    "year",
    "month",
    "day",
    "day_of_week",
    "day_of_year",
    "week_of_year",
    "quarter",
    "is_weekend",
    "month_sin",
    "month_cos",
    "day_sin",
    "day_cos",
    "sales_lag_1",
    "sales_lag_7",
    "sales_lag_14",
    "sales_lag_28",
    "sales_rolling_mean_7",
    "sales_rolling_mean_14",
    "sales_rolling_mean_28"
]

feature_columns = (
    categorical_features + numeric_features
)

print("Features:", len(feature_columns))

Features: 24


In [11]:
model_features = model.feature_names_

if model_features:
    assert model_features == feature_columns, (
        "Model feature names or order do not match!"
    )

print("Model feature configuration verified!")

Model feature configuration verified!


### Build the recursive forecasting function